In [3]:
import torch
import torch.nn.functional as F
import networkx as nx
import math

############################################
# 一些辅助小函数: softmin, softmax
############################################
def softmin(x: torch.Tensor, beta: float = 10.0, dim: int = -1) -> torch.Tensor:
    """
    近似 min(x) 的可微函数:
      softmin_beta(x) = -1/beta * log( sum(exp(-beta*x)) ).
    当 beta 越大, 越逼近真正的 min.
    x  : 任意形状张量
    dim: 在哪个维度上做 softmin
    """
    return -1.0 / beta * torch.logsumexp(-beta * x, dim=dim)

def softmax(x: torch.Tensor, alpha: float = 10.0) -> torch.Tensor:
    """
    近似 max(x) 的可微函数:
      softmax_alpha(x) = 1/alpha * log( sum(exp(alpha*x)) ).
    当 alpha 越大, 越逼近真正的 max.
    x : 任意形状张量(将 flatten 后做全局 max).
    """
    return (1.0 / alpha) * torch.logsumexp(alpha * x.flatten(), dim=0)

############################################
# 用 networkx 计算“真实”直径 (带权)
############################################
def nx_approx_diameter(adj_01: torch.Tensor, W: torch.Tensor) -> float:
    """
    用 networkx 计算图的(带权)直径:
      - adj_01: [N,N], 0~1 的邻接强度(阈值判断是否有边)
      - W:       [N,N], 边权, 若无边则应是 inf
    返回值:
      float型直径; 如果图不连通, 则可能为 inf.
    """
    # 先把数据转到 CPU + numpy
    adj_01_np = adj_01.cpu().numpy()
    W_np = W.cpu().numpy()
    
    N = adj_01_np.shape[0]
    G = nx.Graph()
    G.add_nodes_from(range(N))
    
    # 给定一个小阈值, 超过它视为有边
    threshold = 1e-3
    for i in range(N):
        for j in range(N):
            if i != j:
                if adj_01_np[i, j] > threshold and not math.isinf(W_np[i, j]):
                    G.add_edge(i, j, weight=W_np[i, j])
    
    # 用 all_pairs_dijkstra_path_length 求所有点对距离, 再取最大
    # 若不连通, 则可能出现 inf
    distances = dict(nx.all_pairs_dijkstra_path_length(G, weight='weight'))
    max_dist = 0.0
    for i in distances:
        for j in distances[i]:
            if distances[i][j] > max_dist:
                max_dist = distances[i][j]
    
    # 如果发现有节点距离未覆盖, 说明不连通 => 直径=inf
    if any(len(distances[i]) < N for i in distances):
        return float('inf')
    
    return max_dist

############################################
# 核心函数: 用K-hop + softmin/softmax估计直径, A∈[-1,1]
############################################
def soft_diameter(A: torch.Tensor,
                  W: torch.Tensor,
                  K: int = 3,
                  beta: float = 10.0,
                  alpha: float = 10.0,
                  eps: float = 1e-6) -> torch.Tensor:
    """
    基于 K-hop 近似 + softmin(min) + softmax(max) 的可微直径
    参数:
      A     : [N,N], 范围在 [-1,1], 需要 clamp 或其他方式保证范围
      W     : [N,N], 原始边权(若无边则 inf)
      K     : 最多 hop 数
      beta  : softmin 平滑参数 (越大越近似 min)
      alpha : softmax 平滑参数 (越大越近似 max)
      eps   : 防止除0
    返回:
      标量张量, 近似图的直径
    """
    # 1) 将 A clamp到 [-1,1], 然后映射到 [0,1]
    A_clamped = torch.clamp(A, -1.0, 1.0)
    A_soft = (A_clamped + 1.0) / 2.0  # [0,1]

    # 2) 构造 W_adj: 当 A_soft≈0 => 权重=inf
    W_adj = W / (A_soft + eps)

    # 3) 初始化 P => 1-hop 的最短距离近似
    P = W_adj.clone()

    # 4) 迭代 K-1 次做 softmin => 多步最短距离近似
    for _ in range(K - 1):
        # tmp[i,j,k] = P[i,k] + W_adj[k,j]
        tmp = P.unsqueeze(2) + W_adj.unsqueeze(0)
        # 在 k 维上做 softmin
        P = softmin(tmp, beta=beta, dim=2)
        print(_, P)

    # 5) 用 softmax 近似 max => 近似直径
    D_approx = softmax(P, alpha=alpha)
    return D_approx


############################################
# Demo: 在 GPU 上训练, 用 networkx 对比结果
############################################
if __name__ == "__main__":
    # 0) 选择设备: GPU 或 CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # 1) 初始 A, 大小4x4, 值落在 [-1,1], requires_grad=True
    A_init = [[-1.0,  0.5,  1.0,  -1.0],
              [ 0.5, -1.0,  0.5,  0.5],
              [ 1.0,  0.5, -1.0,  0.5],
              [-1.0,  0.5,  0.5, -1.0]]
    A_torch = torch.tensor(A_init, requires_grad=True, device=device)

    # 2) 原始权重矩阵 W, 若无边 => inf
    W_init = [[float('inf'), 2.0,          3.0,          float('inf')],
              [2.0,          float('inf'), 4.0,          1.0         ],
              [3.0,          4.0,          float('inf'), 2.0         ],
              [float('inf'), 1.0,          2.0,          float('inf')]]
    W_torch = torch.tensor(W_init, device=device)

    # 3) 训练超参
    K     = 3       # 最多 hop
    beta  = 10.0
    alpha = 10.0
    lr    = 0.01
    steps = 20

    optimizer = torch.optim.Adam([A_torch], lr=lr)

    print("Initial A:", A_torch)
    for step in range(steps+1):
        optimizer.zero_grad()
        D_k = soft_diameter(A_torch, W_torch, K=K, beta=beta, alpha=alpha)
        # 我们把近似直径当作loss
        loss = D_k
        loss.backward()
        optimizer.step()

        if step % 5 == 0:
            print(f"Step={step}, ApproxDiameter={loss.item():.4f}")

    # ============ 用 networkx 验证优化后邻接矩阵对应的“真实”直径 =============
    # 把 A_torch 移回 CPU, clamp 到 [-1,1], 然后映射到 [0,1]
    A_optimized = A_torch.detach().cpu().clone()
    A_optimized_clamped = torch.clamp(A_optimized, -1, 1)
    A_optimized_01 = (A_optimized_clamped + 1.0) / 2.0  # [0,1]

    # 用 networkx 计算直径
    diameter_nx = nx_approx_diameter(A_optimized_01, W_torch.cpu())
    print(f"\n[NetworkX] Computed diameter = {diameter_nx}")
    
    print("Optimized A in [-1,1]:")
    print(A_optimized)


Using device: cuda
Initial A: tensor([[-1.0000,  0.5000,  1.0000, -1.0000],
        [ 0.5000, -1.0000,  0.5000,  0.5000],
        [ 1.0000,  0.5000, -1.0000,  0.5000],
        [-1.0000,  0.5000,  0.5000, -1.0000]], device='cuda:0',
       requires_grad=True)
0 tensor([[   inf, 4.0000, 5.6632,    inf],
        [5.3298,    inf, 7.9965, 2.6667],
        [5.6632, 6.6667,    inf, 4.0000],
        [   inf, 2.6667, 5.3298,    inf]], device='cuda:0',
       grad_fn=<MulBackward0>)
1 tensor([[    inf,  5.3333,  8.3263,     inf],
        [ 7.9930,     inf, 10.6596,  4.0000],
        [ 8.3263,  8.0000,     inf,  5.3333],
        [    inf,  4.0000,  7.9930,     inf]], device='cuda:0',
       grad_fn=<MulBackward0>)
Step=0, ApproxDiameter=inf
0 tensor([[nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan]], device='cuda:0', grad_fn=<MulBackward0>)
1 tensor([[nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
    